In [1]:
import os
from flask import Flask, jsonify, request
from litellm import completion_with_retries

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

app = Flask(__name__)

@app.route("/", methods=["GET"])
def hello():
    return jsonify(message="Hello, Flask!")

@app.route("/chat/completions", methods=["POST"])
def api_completion():
    data = request.get_json()

    if not data:
        return jsonify({"error": "Request body must be JSON"}), 400

    data["max_tokens"] = 256

    try:
        response = completion_with_retries(**data)

        if hasattr(response, "model_dump"):
            return jsonify(response.model_dump())

        return jsonify(response)

    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    from waitress import serve
    serve(app, host="0.0.0.0", port=4000, threads=500)